In [1]:
pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import pipeline
import pandas as pd
import re

# Initialize NER pipeline with proper token merging
pipe = pipeline("token-classification", 
                model="d4data/biomedical-ner-all",
                aggregation_strategy='first')

# Load and preprocess dataset
data = pd.read_csv('DATASET.csv', encoding='latin1')
data['Symptom_lower'] = data['Symptom'].str.strip().str.lower()

# Enhanced severity keywords
SEVERITY_KEYWORDS = {
    'normal': ['mild', 'occasional', 'controlled', 'temporary', 'fleeting'],
    'moderate': ['moderate', 'persistent', 'frequent', 'prolonged', 'bloating'],
    'severe': ['severe', 'extreme', 'crushing', 'radiating', 'blood', 'inability', 
              'sweating', 'vomiting', 'fever', 'swelling', 'radiates', 'persistent >3 days']
}

def merge_entities(entities, text):
    """Merge entities using dataset symptom names for multi-word matching"""
    merged = []
    detected = set()
    text_lower = text.lower()
    
    # Prioritize longer symptom names first
    symptoms_sorted = data['Symptom'].str.lower().sort_values(key=lambda x: x.str.len(), ascending=False).tolist()
    
    
    for symptom in symptoms_sorted:
        if symptom in text_lower and symptom not in detected:
            start = text_lower.find(symptom)
            end = start + len(symptom)
            merged.append({
                'word': text[start:end],
                'start': start,
                'end': end,
                'entity_group': 'Sign_symptom'
            })
            detected.add(symptom)
    
    return merged

def get_risk_score(symptom, condition):
    symptom_clean = symptom.strip().lower()
    matches = data[
        data['Symptom_lower'].str.contains(rf'\b{symptom_clean}\b', regex=True, flags=re.IGNORECASE)
    ]

    if not matches.empty:
        matches = matches[matches['Condition'] == condition]
        return matches['Risk Score'].values[0] if not matches.empty else None
    return None

def determine_condition(text, entity):
    context_window = re.findall(r'\w+', text[max(0, entity['start']-20):entity['end']+20])
    context = ' '.join(context_window).lower()
    
    for level, keywords in SEVERITY_KEYWORDS.items():
        if any(k in context for k in keywords):
            return level.capitalize()
    return 'Normal'

    # Rest remains the same...

def calculate_risk_score(text, entities):
    merged_entities = merge_entities(entities, text)  # Pass text to merger

    score = 0
    for ent in merged_entities:
        if ent['entity_group'] == "Sign_symptom":
            symptom = ent['word'].strip().lower()
            condition = determine_condition(text, ent)
            risk = get_risk_score(symptom, condition)
            if risk:
                score += risk
    return score

# Test cases
patients = [
    ["28-year-old woman with mild sneezing and occasional runny nose"],
    ["45-year-old patient with severe chest pain radiating to the arm"],
    ["60-year-old woman with persistent cough and blood in stool"],
    ["21-year-old male with irregular and sharp chest pain, with slight radiation to the back."]
]

for idx, patient in enumerate(patients):
    print(f"\nPatient {idx+1}:")
    total = 0
    for response in patient:
        entities = pipe(response)
        score = calculate_risk_score(response, entities)
        total += score
        print(f"Response: {response}")
        # Pass both entities AND text to merge_entities
        merged_symptoms = merge_entities(entities, response)  
        print(f"Detected Symptoms: {[e['word'] for e in merged_symptoms if e['entity_group']=='Sign_symptom']}")
        print(f"Risk Score: {score}")
    print(f"Total Risk Score: {total}\n{'-'*40}")

Groq client initialized successfully!
Dr. AI: Welcome to AI Medical Consultancy. Let's start with some basic information.

Dr. AI: Hello, I understand you're here because of a fever. Can you tell me how long you have been experiencing this fever?

Dr. AI: Thank you for sharing that information. To help me better understand your situation, I would like to ask: Have you noticed any other symptoms accompanying the fever, such as cough, sore throat, body aches, or difficulty breathing?

Dr. AI: I understand that you have not noticed any other symptoms accompanying the fever. To further narrow down the possible causes, I would like to ask: Have you recently traveled to any areas with known outbreaks of infectious diseases, such as malaria or dengue fever?

Dr. AI: Given that you have not traveled to areas with known outbreaks of infectious diseases, I would like to ask: Have you been in close contact with anyone who has a confirmed viral illness, such as COVID-19, influenza, or mononucleosi

In [3]:
from transformers import pipeline
import pandas as pd
import re

# Initialize NER pipeline with proper token merging
pipe = pipeline("token-classification", 
                model="d4data/biomedical-ner-all",
                aggregation_strategy='first')

# Load and preprocess dataset
data = pd.read_csv('DATASET.csv', encoding='latin1')
data['Symptom_lower'] = data['Symptom'].str.strip().str.lower()

# Enhanced severity keywords
SEVERITY_KEYWORDS = {
    'normal': ['mild', 'occasional', 'controlled', 'temporary', 'fleeting'],
    'moderate': ['moderate', 'persistent', 'frequent', 'prolonged', 'bloating'],
    'severe': ['severe', 'extreme', 'crushing', 'radiating', 'blood', 'inability', 
              'sweating', 'vomiting', 'fever', 'swelling', 'radiates', 'persistent >3 days']
}

def merge_entities(entities, text):
    """Merge entities using dataset symptom names for multi-word matching"""
    merged = []
    detected = set()
    text_lower = text.lower()
    
    # Prioritize longer symptom names first
    symptoms_sorted = data['Symptom'].str.lower().sort_values(key=lambda x: x.str.len(), ascending=False).tolist()
    
    
    for symptom in symptoms_sorted:
        if symptom in text_lower and symptom not in detected:
            start = text_lower.find(symptom)
            end = start + len(symptom)
            merged.append({
                'word': text[start:end],
                'start': start,
                'end': end,
                'entity_group': 'Sign_symptom'
            })
            detected.add(symptom)
    
    return merged

def get_risk_score(symptom, condition):
    symptom_clean = symptom.strip().lower()
    matches = data[
        data['Symptom_lower'].str.contains(rf'\b{symptom_clean}\b', regex=True, flags=re.IGNORECASE)
    ]

    if not matches.empty:
        matches = matches[matches['Condition'] == condition]
        return matches['Risk Score'].values[0] if not matches.empty else None
    return None

def determine_condition(text, entity):
    context_window = re.findall(r'\w+', text[max(0, entity['start']-20):entity['end']+20])
    context = ' '.join(context_window).lower()
    
    for level, keywords in SEVERITY_KEYWORDS.items():
        if any(k in context for k in keywords):
            return level.capitalize()
    return 'Normal'

    # Rest remains the same...

def calculate_risk_score(text, entities):
    merged_entities = merge_entities(entities, text)  # Pass text to merger

    score = 0
    for ent in merged_entities:
        if ent['entity_group'] == "Sign_symptom":
            symptom = ent['word'].strip().lower()
            condition = determine_condition(text, ent)
            risk = get_risk_score(symptom, condition)
            if risk:
                score += risk
    return score

# Test cases
patients = [
    ["28-year-old woman with mild sneezing and occasional runny nose"],
    ["45-year-old patient with severe chest pain radiating to the arm"],
    ["60-year-old woman with persistent cough and blood in stool"],
    ["21-year-old male with irregular and sharp chest pain, with slight radiation to the back."]
]

for idx, patient in enumerate(patients):
    print(f"\nPatient {idx+1}:")
    total = 0
    for response in patient:
        entities = pipe(response)
        score = calculate_risk_score(response, entities)
        total += score
        print(f"Response: {response}")
        # Pass both entities AND text to merge_entities
        merged_symptoms = merge_entities(entities, response)  
        print(f"Detected Symptoms: {[e['word'] for e in merged_symptoms if e['entity_group']=='Sign_symptom']}")
        print(f"Risk Score: {score}")
    print(f"Total Risk Score: {total}\n{'-'*40}")

ModuleNotFoundError: No module named 'transformers'